<a href="https://colab.research.google.com/github/dayanakumar-IT/R26-DS-010-Intelligent-Care-Support/blob/caregiver-deterioration-ai/Final_DATASET_Preprocessing_O_Audio_RAVEDESS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---


============================================================
NOTEBOOK: 03_Audio_Model_RAVDESS.ipynb
Run this on your MAIN laptop in a new Colab notebook
============================================================


---



In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH    = '/content/drive/MyDrive/CareSense_Research'
AUDIO_PATH   = os.path.join(BASE_PATH, 'Audio_Raw', 'RAVDESS')
PROCESSED    = os.path.join(BASE_PATH, 'Processed')
FIGURES      = os.path.join(BASE_PATH, 'Figures')

os.makedirs(AUDIO_PATH, exist_ok=True)
os.makedirs(PROCESSED,  exist_ok=True)

print("✓ Drive mounted")
print(f"  Audio path: {AUDIO_PATH}")

Mounted at /content/drive
✓ Drive mounted
  Audio path: /content/drive/MyDrive/CareSense_Research/Audio_Raw/RAVDESS


In [2]:
# =============================================================================
# CELL 2 — Install Libraries
# =============================================================================

get_ipython().system('pip install librosa -q')
get_ipython().system('pip install xgboost imbalanced-learn -q')

import librosa
import numpy as np
import pandas as pd
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics         import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score
)
from imblearn.over_sampling  import SMOTE

print("✓ All libraries ready")

✓ All libraries ready


In [6]:
# =============================================================================
# CELL 3 — FIXED VERSION
# Run this instead of the original Cell 3
# =============================================================================

import os

# Check if already in Drive
actor_folders = [
    f for f in os.listdir(AUDIO_PATH)
    if f.startswith('Actor_')
] if os.path.exists(AUDIO_PATH) else []

if len(actor_folders) == 24:
    print(f"✓ RAVDESS already in Drive ({len(actor_folders)} actor folders)")
else:
    print("Downloading RAVDESS from Zenodo...")
    get_ipython().system(
        'wget -q --show-progress '
        '"https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip" '
        '-O /content/RAVDESS.zip'
    )
    print("✓ Download complete")

    print("Unzipping...")
    get_ipython().system('unzip -q /content/RAVDESS.zip -d /content/RAVDESS_extracted/')
    print("✓ Unzipped")

    # Find what was actually extracted
    print("\nChecking extracted contents...")
    for root, dirs, files in os.walk('/content/RAVDESS_extracted/'):
        level = root.replace('/content/RAVDESS_extracted/', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 2:
            for f in files[:3]:
                print(f"{indent}  {f}")
        break  # only show top level first

    # Find Actor folders wherever they are
    actor_source = None
    for root, dirs, files in os.walk('/content/RAVDESS_extracted/'):
        actor_dirs = [d for d in dirs if d.startswith('Actor_')]
        if len(actor_dirs) > 0:
            actor_source = root
            print(f"\n✓ Found {len(actor_dirs)} Actor folders at: {root}")
            break

    if actor_source is None:
        print("✗ Could not find Actor folders. Listing all extracted content:")
        get_ipython().system('ls -la /content/RAVDESS_extracted/')
        get_ipython().system('find /content/RAVDESS_extracted/ -name "*.wav" | head -5')
    else:
        print(f"Copying to Drive...")
        get_ipython().system(f'cp -r "{actor_source}"/Actor_* "{AUDIO_PATH}/"')
        print("✓ Copied to Drive")

# Verify
actor_folders = [
    f for f in os.listdir(AUDIO_PATH)
    if f.startswith('Actor_')
] if os.path.exists(AUDIO_PATH) else []

print(f"\n  Actor folders found: {len(actor_folders)}")
print(f"  Expected: 24 actors")

total_files = 0
for actor in actor_folders:
    actor_path = os.path.join(AUDIO_PATH, actor)
    if os.path.isdir(actor_path):
        total_files += len([f for f in os.listdir(actor_path) if f.endswith('.wav')])

print(f"  Total WAV files: {total_files}")
print(f"  Expected: ~1,440 speech files")

/content/RAVDESS.zi 100%[===================>] 198.81M  29.6MB/s    in 7.7s    
✓ Download complete
Unzipping...
replace /content/RAVDESS_extracted/Actor_01/03-01-01-01-01-01-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-01-01-01-02-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-01-01-02-01-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-01-01-02-02-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-02-01-01-01-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-02-01-01-02-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-02-01-02-01-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/RAVDESS_extracted/Actor_01/03-01-02-01-02-02-01.wav? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
✓ Unzip

In [5]:
import os
for root, dirs, files in os.walk('/content/RAVDESS_extracted/'):
    print(root)
    print("  dirs:", dirs[:5])
    print("  files:", files[:3])
    break


/content/RAVDESS_extracted/
  dirs: ['Actor_21', 'Actor_18', 'Actor_04', 'Actor_22', 'Actor_19']
  files: []


In [7]:
# =============================================================================
# CELL 4 — Parse RAVDESS Filenames and Create Labels
#
# RAVDESS FILENAME FORMAT: 03-01-05-01-02-01-12.wav
# Position (index):
#   0: Modality       (03=audio-only)
#   1: Vocal channel  (01=speech)
#   2: Emotion        (01=neutral,02=calm,03=happy,04=sad,
#                      05=angry,06=fearful,07=disgust,08=surprised)
#   3: Intensity      (01=normal,02=strong)
#   4: Statement      (01,02)
#   5: Repetition     (01,02)
#   6: Actor          (01-24, odd=male, even=female)
#
# STRESS MAPPING:
# Stressed (1):     angry(05), fearful(06), disgusted(07)
# Not stressed (0): neutral(01), calm(02), happy(03)
# Excluded:         sad(04), surprised(08) — ambiguous arousal
#
# WHY THIS MAPPING:
# Psychological stress involves high arousal and negative valence.
# Angry, fearful, and disgusted match this profile.
# Calm, neutral, and happy are clearly non-stressed states.
# Sad has negative valence but low arousal — different from stress.
# =============================================================================

from pathlib import Path

STRESS_MAP = {
    '01': 0,    # neutral → not stressed
    '02': 0,    # calm → not stressed
    '03': 0,    # happy → not stressed
    '04': None, # sad → excluded (low arousal, different from stress)
    '05': 1,    # angry → stressed
    '06': 1,    # fearful → stressed
    '07': 1,    # disgusted → stressed
    '08': None, # surprised → excluded (ambiguous)
}

EMOTION_NAMES = {
    '01': 'neutral', '02': 'calm',    '03': 'happy',
    '04': 'sad',     '05': 'angry',   '06': 'fearful',
    '07': 'disgust', '08': 'surprised'
}

records = []

for actor_folder in sorted(os.listdir(AUDIO_PATH)):
    actor_path = os.path.join(AUDIO_PATH, actor_folder)
    if not os.path.isdir(actor_path):
        continue

    actor_num = actor_folder.split('_')[-1]  # e.g. "01"
    gender    = 'female' if int(actor_num) % 2 == 0 else 'male'

    for wav_file in sorted(os.listdir(actor_path)):
        if not wav_file.endswith('.wav'):
            continue

        parts = Path(wav_file).stem.split('-')
        if len(parts) != 7:
            continue

        modality = parts[0]
        if modality != '03':  # audio-only
            continue

        emotion_code = parts[2]
        label        = STRESS_MAP.get(emotion_code)

        if label is None:
            continue  # exclude sad and surprised

        records.append({
            'filepath'    : os.path.join(actor_path, wav_file),
            'actor_id'    : actor_num,
            'gender'      : gender,
            'emotion_code': emotion_code,
            'emotion_name': EMOTION_NAMES.get(emotion_code, 'unknown'),
            'intensity'   : parts[3],
            'stress_label': label
        })

metadata_df = pd.DataFrame(records)

print("=" * 50)
print("RAVDESS DATASET OVERVIEW")
print("=" * 50)
print(f"\n  Total files (after excluding sad+surprised): {len(metadata_df)}")
print(f"\n  Stress label distribution:")
counts = metadata_df['stress_label'].value_counts().sort_index()
for label, count in counts.items():
    name = 'Not Stressed' if label == 0 else 'Stressed'
    pct  = count / len(metadata_df) * 100
    print(f"  {name} ({label}): {count} files ({pct:.1f}%)")

print(f"\n  Emotion breakdown:")
print(metadata_df.groupby(['emotion_name', 'stress_label']).size().to_string())

print(f"\n  Gender distribution:")
print(metadata_df['gender'].value_counts().to_string())

print(f"\n  Actors: {metadata_df['actor_id'].nunique()} (24 expected)")

RAVDESS DATASET OVERVIEW

  Total files (after excluding sad+surprised): 1056

  Stress label distribution:
  Not Stressed (0): 480 files (45.5%)
  Stressed (1): 576 files (54.5%)

  Emotion breakdown:
emotion_name  stress_label
angry         1               192
calm          0               192
disgust       1               192
fearful       1               192
happy         0               192
neutral       0                96

  Gender distribution:
gender
male      528
female    528

  Actors: 24 (24 expected)


In [8]:
# =============================================================================
# CELL 5 — Extract Acoustic Features
#
# FEATURES EXTRACTED (language-independent):
# These features capture HOW voice sounds physically,
# not WHAT words are spoken. Works in any language.
#
# MFCC (26 features):
#   Mel-frequency cepstral coefficients — captures vocal tract
#   shape. Changes with stress because muscles tighten.
#   13 coefficients × mean + std = 26 features
#
# Pitch/F0 (3 features):
#   Fundamental frequency — stressed speech has higher,
#   more variable pitch. Mean, std, range.
#
# Energy/RMS (2 features):
#   Signal power — stressed speech is louder, more forceful.
#   Mean and std.
#
# Zero Crossing Rate (2 features):
#   How often signal crosses zero — captures voiced/unvoiced ratio.
#   Mean and std.
#
# Spectral Centroid (1 feature):
#   Where energy concentrates in frequency.
#   Stressed voice shifts energy upward.
#
# Total: 34 language-independent features
# =============================================================================

from scipy import stats as scipy_stats

def extract_features(filepath, duration=3.0, sr=22050):
    """
    Extract 34 language-independent acoustic features from WAV.
    Same function used in production for nurse voice check-ins.
    """
    try:
        audio, _ = librosa.load(filepath, sr=sr, duration=duration)

        if len(audio) < sr * 0.5:  # less than 0.5 seconds
            return None

        features = {}

        # ── MFCC (13 × mean + std = 26 features) ─────────────────
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        for i in range(13):
            features[f'mfcc_{i+1}_mean'] = float(np.mean(mfcc[i]))
            features[f'mfcc_{i+1}_std']  = float(np.std(mfcc[i]))

        # ── Pitch / F0 (3 features) ───────────────────────────────
        f0 = librosa.yin(
            audio,
            fmin=librosa.note_to_hz('C2'),  # 65 Hz
            fmax=librosa.note_to_hz('C7')   # 2093 Hz
        )
        f0_voiced = f0[f0 > 0]
        if len(f0_voiced) > 0:
            features['pitch_mean']  = float(np.mean(f0_voiced))
            features['pitch_std']   = float(np.std(f0_voiced))
            features['pitch_range'] = float(f0_voiced.max() - f0_voiced.min())
        else:
            features['pitch_mean']  = 0.0
            features['pitch_std']   = 0.0
            features['pitch_range'] = 0.0

        # ── RMS Energy (2 features) ───────────────────────────────
        rms = librosa.feature.rms(y=audio)[0]
        features['energy_mean'] = float(np.mean(rms))
        features['energy_std']  = float(np.std(rms))

        # ── Zero Crossing Rate (2 features) ──────────────────────
        zcr = librosa.feature.zero_crossing_rate(audio)[0]
        features['zcr_mean'] = float(np.mean(zcr))
        features['zcr_std']  = float(np.std(zcr))

        # ── Spectral Centroid (1 feature) ─────────────────────────
        centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
        features['spectral_centroid_mean'] = float(np.mean(centroid))

        return features

    except Exception:
        return None


# Run extraction on all files
print("Extracting features from RAVDESS...")
print(f"Total files: {len(metadata_df)}")
print("Expected time: 10-15 minutes\n")

all_features = []
failed       = 0

for idx, row in metadata_df.iterrows():
    features = extract_features(row['filepath'])

    if features is not None:
        features['actor_id']    = row['actor_id']
        features['gender']      = row['gender']
        features['emotion_name']= row['emotion_name']
        features['stress_label']= row['stress_label']
        all_features.append(features)
    else:
        failed += 1

    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx+1}/{len(metadata_df)} files...")

feature_df = pd.DataFrame(all_features)

print(f"\n✓ Feature extraction complete")
print(f"  Successful: {len(feature_df)}")
print(f"  Failed:     {failed}")
print(f"  Features:   {len([c for c in feature_df.columns if c not in ['actor_id','gender','emotion_name','stress_label']])}")
print(f"\n  Label distribution:")
print(feature_df['stress_label'].value_counts().sort_index().to_string())

# Save features
feat_path = os.path.join(PROCESSED, 'audio_features_ravdess.csv')
feature_df.to_csv(feat_path, index=False)
print(f"\n✓ Features saved: {feat_path}")

Extracting features from RAVDESS...
Total files: 1056
Expected time: 10-15 minutes

  Processed 100/1056 files...
  Processed 200/1056 files...
  Processed 300/1056 files...
  Processed 400/1056 files...
  Processed 500/1056 files...
  Processed 600/1056 files...
  Processed 700/1056 files...
  Processed 800/1056 files...
  Processed 900/1056 files...
  Processed 1000/1056 files...

✓ Feature extraction complete
  Successful: 1056
  Failed:     0
  Features:   34

  Label distribution:
stress_label
0    480
1    576

✓ Features saved: /content/drive/MyDrive/CareSense_Research/Processed/audio_features_ravdess.csv


In [9]:
# =============================================================================
# CELL 6 — Actor-Independent Cross-Validation and Model Training
#
# WHY ACTOR-INDEPENDENT (not random split):
# Same reason as LOSO in physiological data.
# If Actor 1 appears in both train and test, the model
# may learn Actor 1's voice rather than stress patterns.
# Actor-independent split ensures the model learns features
# that generalise to NEW speakers — equivalent to real deployment
# where the nurse was never in training data.
#
# METHOD: StratifiedKFold on actors
# Each fold leaves out ~4-5 actors as test subjects.
# Stress labels are stratified to maintain balance per fold.
# =============================================================================

# Reload if needed
try:
    _ = feature_df.shape
except NameError:
    feature_df = pd.read_csv(os.path.join(PROCESSED, 'audio_features_ravdess.csv'))

FEATURE_COLS = [
    c for c in feature_df.columns
    if c not in ['actor_id', 'gender', 'emotion_name', 'stress_label']
]

X      = feature_df[FEATURE_COLS].values.astype(float)
y      = feature_df['stress_label'].values.astype(int)
actors = feature_df['actor_id'].values

print("=" * 55)
print("ACTOR-INDEPENDENT CROSS-VALIDATION")
print("=" * 55)
print(f"\n  Features: {X.shape[1]}")
print(f"  Samples:  {X.shape[0]}")
print(f"  Actors:   {len(np.unique(actors))}")

MODELS = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    )
}

# Use GroupKFold on actors for proper speaker-independence
from sklearn.model_selection import GroupKFold

gkf      = GroupKFold(n_splits=5)
all_res  = []
fold_num = 0

print("\n  Running 5-fold actor-independent cross-validation...\n")

for train_idx, test_idx in gkf.split(X, y, groups=actors):

    fold_num += 1
    test_actors_fold = np.unique(actors[test_idx])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Scale — fit on train only
    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_train)
    X_te_sc  = scaler.transform(X_test)

    # SMOTE on training only
    try:
        unique, counts = np.unique(y_train, return_counts=True)
        if counts.min() >= 2:
            smote    = SMOTE(k_neighbors=min(5, counts.min()-1), random_state=42)
            X_tr_bal, y_tr_bal = smote.fit_resample(X_tr_sc, y_train)
        else:
            X_tr_bal, y_tr_bal = X_tr_sc, y_train
    except Exception:
        X_tr_bal, y_tr_bal = X_tr_sc, y_train

    print(f"  Fold {fold_num}/5 | Test actors: {test_actors_fold}",
          end='', flush=True)

    for model_name, model in MODELS.items():
        model.fit(X_tr_bal, y_tr_bal)
        y_pred       = model.predict(X_te_sc)
        y_pred_proba = model.predict_proba(X_te_sc)

        binary_f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        accuracy  = accuracy_score(y_test, y_pred)
        try:
            auc = roc_auc_score(y_test, y_pred_proba[:, 1])
        except Exception:
            auc = float('nan')

        all_res.append({
            'fold'     : fold_num,
            'model'    : model_name,
            'binary_f1': binary_f1,
            'accuracy' : accuracy,
            'roc_auc'  : auc
        })

    xgb_f1 = next(r['binary_f1'] for r in all_res
                   if r['fold']==fold_num and r['model']=='XGBoost')
    rf_f1  = next(r['binary_f1'] for r in all_res
                   if r['fold']==fold_num and r['model']=='Random Forest')
    print(f" | RF:{rf_f1:.3f}  XGB:{xgb_f1:.3f}")

results_df = pd.DataFrame(all_res)

print("\n" + "=" * 55)
print("AUDIO MODEL RESULTS")
print("=" * 55)
print(f"\n  {'Model':<22} {'Binary F1':<14} {'ROC-AUC':<12} {'Accuracy'}")
print("  " + "-" * 58)

audio_summary = {}
for model_name in MODELS.keys():
    mdf      = results_df[results_df['model'] == model_name]
    mean_f1  = mdf['binary_f1'].mean()
    std_f1   = mdf['binary_f1'].std()
    mean_auc = mdf['roc_auc'].mean()
    mean_acc = mdf['accuracy'].mean()
    audio_summary[model_name] = {
        'mean_f1': mean_f1, 'std_f1': std_f1,
        'mean_auc': mean_auc
    }
    print(f"  {model_name:<22} {mean_f1:.3f}±{std_f1:.3f}   "
          f"{mean_auc:.3f}       {mean_acc:.3f}")

best_audio_model = max(audio_summary, key=lambda m: audio_summary[m]['mean_f1'])
print(f"\n  ✓ Best model: {best_audio_model}")
print(f"  Binary F1: {audio_summary[best_audio_model]['mean_f1']:.3f}")



ACTOR-INDEPENDENT CROSS-VALIDATION

  Features: 34
  Samples:  1056
  Actors:   24

  Running 5-fold actor-independent cross-validation...

  Fold 1/5 | Test actors: ['04' '09' '14' '19' '24'] | RF:0.800  XGB:0.797
  Fold 2/5 | Test actors: ['03' '08' '13' '18' '23'] | RF:0.749  XGB:0.777
  Fold 3/5 | Test actors: ['02' '07' '12' '17' '22'] | RF:0.805  XGB:0.789
  Fold 4/5 | Test actors: ['01' '06' '11' '16' '21'] | RF:0.770  XGB:0.753
  Fold 5/5 | Test actors: ['05' '10' '15' '20'] | RF:0.726  XGB:0.691

AUDIO MODEL RESULTS

  Model                  Binary F1      ROC-AUC      Accuracy
  ----------------------------------------------------------
  Logistic Regression    0.780±0.036   0.826       0.761
  Random Forest          0.770±0.033   0.808       0.742
  XGBoost                0.761±0.043   0.788       0.732

  ✓ Best model: Logistic Regression
  Binary F1: 0.780


In [10]:
# =============================================================================
# CELL 7 — Train Final Model and Save
# =============================================================================

print("\nTraining final audio model on all data...")

final_scaler = StandardScaler()
X_scaled_all = final_scaler.fit_transform(X)

try:
    smote_final = SMOTE(k_neighbors=5, random_state=42)
    X_bal, y_bal = smote_final.fit_resample(X_scaled_all, y)
    print(f"SMOTE: {len(y)} → {len(y_bal)} samples")
except Exception:
    X_bal, y_bal = X_scaled_all, y

final_audio_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1
)
final_audio_model.fit(X_bal, y_bal)

# Save
audio_model_path  = os.path.join(PROCESSED, 'audio_model.pkl')
audio_scaler_path = os.path.join(PROCESSED, 'audio_scaler.pkl')
audio_feats_path  = os.path.join(PROCESSED, 'audio_feature_names.json')

joblib.dump(final_audio_model, audio_model_path)
joblib.dump(final_scaler, audio_scaler_path)

with open(audio_feats_path, 'w') as f:
    json.dump(FEATURE_COLS, f)

audio_results_path = os.path.join(PROCESSED, 'audio_results.csv')
results_df.to_csv(audio_results_path, index=False)

print(f"\n✓ audio_model.pkl saved")
print(f"✓ audio_scaler.pkl saved")
print(f"✓ audio_feature_names.json saved ({len(FEATURE_COLS)} features)")
print(f"✓ audio_results.csv saved")

print(f"""
======================================================
AUDIO MODEL COMPLETE
======================================================
Dataset:    RAVDESS (Livingstone & Russo 2018)
Actors:     24 (12 male, 12 female)
Validation: 5-fold actor-independent cross-validation
Best model: {best_audio_model}
Binary F1:  {audio_summary[best_audio_model]['mean_f1']:.3f} ± {audio_summary[best_audio_model]['std_f1']:.3f}
ROC-AUC:    {audio_summary[best_audio_model]['mean_auc']:.3f}

Stressed class:     angry + fearful + disgusted
Not stressed class: neutral + calm + happy

→ Next: Run fusion notebook (Cell 15 in main notebook)
""")


Training final audio model on all data...
SMOTE: 1056 → 1152 samples

✓ audio_model.pkl saved
✓ audio_scaler.pkl saved
✓ audio_feature_names.json saved (34 features)
✓ audio_results.csv saved

AUDIO MODEL COMPLETE
Dataset:    RAVDESS (Livingstone & Russo 2018)
Actors:     24 (12 male, 12 female)
Validation: 5-fold actor-independent cross-validation
Best model: Logistic Regression
Binary F1:  0.780 ± 0.036
ROC-AUC:    0.826
 
Stressed class:     angry + fearful + disgusted
Not stressed class: neutral + calm + happy
 
→ Next: Run fusion notebook (Cell 15 in main notebook)

